# Results for model: mistralai/mistral-7b-instruct-v0.3

In [ ]:
import polars as pl
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

pl.init_cpp()  # Initialize Polars C++ engine

data = pl.read_parquet('./jane_street_data/train.parquet')

rolling_top_quantile = data.groupby('date_id') \
                           .agg({' feature_00': pl.col('percentile_cont(0.15)' if 'feature_00' == col else col) for col in data.columns})
rolling_top_quantile = rolling_top_quantile.sort('date_id', on='date_id').drop_duplicates('date_id')

features = ['feature_00', *[f'top_quantile_{i}' for i in range(1, 7)]]  # Assuming 5 additional columns for top quantile (15%, 25%, 50%, 75%, 95%)
X = rolling_top_quantile.select(features)
y = rolling_top_quantile['responder_6']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = XGBRegressor(objective='reg:squarederror', colsample_bytree=0.3, learning_rate=0.01)
model.fit(X_train, y_train)

predictions = model.predict(X_test)